# Historical inference candidate
Original code from Model_Tensorflow/model_testing.ipynb. Output and execution metadata removed on 2026-10-01. Original paths/code remain; inspect rather than running unreviewed. Uses the candidate 27112023184806 checkpoint; epoch/result association is unverified. The supported explicit-path workflow is under src/.


# Model Testing 

In [ ]:
import os
import cv2
import tifffile 
import numpy as np
from tqdm import tqdm
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from model import load_model, build_model
from data_processing import black_to_transparent, overlay_transparent, create_mask, create_visible_mask, cut_image, cut_mask, cut_img, cut_msk
plt.rcParams['figure.dpi'] = 300
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
%env SM_FRAMEWORK=tf.keras

In [ ]:
gpu_devices = tf.config.experimental.list_physical_devices('GPU')
for device in gpu_devices:
    tf.config.experimental.set_memory_growth(device, True)

In [ ]:
m = build_model()
model_path = '27112023184806/best_model.ckpt'
model = load_model(m, model_path)

## Model Visualization 

### Xview2 Images
* The following are images from Xview2 dataset that are deemed decent representation of the model performance.
* Images from "hold" & "test" folders were not used in the training.
* under each image a comment guiding its use case. `(image slices to be used - classes that exist)`

In [ ]:
# image_path = '../geotiffs/test/images/hurricane-harvey_00000096_post_disaster.tif' 
# all - major & few destroyed 
image_path = '../geotiffs/test/images/hurricane-harvey_00000127_pre_disaster.tif' 
# all slices - major 
# image_path = '../geotiffs/test/images/hurricane-harvey_00000112_post_disaster.tif' 
# 3 & 4 - major & minor 
# image_path = '../geotiffs/test/images/palu-tsunami_00000004_post_disaster.tif' 
# 1 - major & undamaged 
# image_path = '../geotiffs/test/images/santa-rosa-wildfire_00000012_post_disaster.tif' 
# 2 - nodmg & destroyed
# image_path = '../geotiffs/hold/images/guatemala-volcano_00000004_pre_disaster.tif'
# 1 - all classes 
# image_path = '../geotiffs/hold/images/hurricane-florence_00000021_post_disaster.tif'
# 1 - all classes 
# image_path = '../geotiffs/hold/images/hurricane-harvey_00000117_post_disaster.tif'
# 1 - most classes 
# image_path = '../geotiffs/hold/images/hurricane-harvey_00000149_post_disaster.tif'
# 3 - minor & major dmg
# image_path = '../geotiffs/test/images/mexico-earthquake_00000076_post_disaster.tif'
# 3 - minor & major classes 
# image_path = '../geotiffs/tier3/images/joplin-tornado_00000000_post_disaster.tif'

### Using Cut Technique 

In [ ]:
images = cut_image(image_path)
masks = cut_mask(image_path.replace('images','labels').replace('.tif','.json'))
n = 1 # slice number to be used
y = model.predict(images[n])

y = np.squeeze(y, axis=0)
y = np.argmax(y, axis=-1)
image = np.squeeze(images[n], axis=0)

# Applying the same annotation coloring as the original mask
y = y.astype(np.uint8)
y = cv2.cvtColor(y, cv2.COLOR_GRAY2RGB)
y[np.all(y == (1,1,1), axis=-1)] = (51,255,255)
y[np.all(y == (2,2,2), axis=-1)] = (255,255,0)
y[np.all(y == (3,3,3), axis=-1)] = (255,128,0)
y[np.all(y == (4,4,4), axis=-1)] = (255,0,0)

In [ ]:
fig1, ax = plt.subplots(1, 3)

ax[0].imshow(image)
ax[0].set_title("Image")
ax[0].axis('off')

ax[1].imshow(y)
ax[1].set_title("Predicted")
ax[1].axis('off')

ax[2].imshow(masks[n])
ax[2].set_title("Actual")
ax[2].axis('off')

# color value mapping
label_to_rgb = {
    'Undamaged': (51,255,255),
    'Minor Dmg': (255,255,0),
    'Major Dmg': (255,128,0),
    'Destroyed': (255,0,0),
}
# Creating patches for the legend with RGB colors
patches = [mpatches.Patch(color=np.array(rgb)/255, label=label) for label, rgb in label_to_rgb.items()]
# Adding the legend to the third subplot
ax[2].legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### Using Resize Technique 

In [ ]:
resized_img = cv2.resize(tifffile.imread(image_path).astype('uint8'), (512, 512), interpolation = cv2.INTER_LINEAR)
resized_img = np.expand_dims(resized_img/255, axis=0)

resized_msk = create_visible_mask(image_path.replace(".tif", ".json").replace("images", "labels"))

y_resized = model.predict(resized_img)

y_resized = np.squeeze(y_resized, axis=0)
y_resized = np.argmax(y_resized, axis=-1)
resized_img = np.squeeze(resized_img, axis=0)

# Applying the same annotation coloring as the original mask
y_resized = y_resized.astype(np.uint8)
y_resized = cv2.cvtColor(y_resized, cv2.COLOR_GRAY2RGB)
y_resized[np.all(y_resized == (1,1,1), axis=-1)] = (51,255,255)
y_resized[np.all(y_resized == (2,2,2), axis=-1)] = (255,255,0)
y_resized[np.all(y_resized == (3,3,3), axis=-1)] = (255,128,0)
y_resized[np.all(y_resized == (4,4,4), axis=-1)] = (255,0,0)

In [ ]:
fig2, ax = plt.subplots(1, 3)

ax[0].imshow(resized_img)
ax[0].set_title("Image")
ax[0].axis('off')

ax[1].imshow(y_resized)
ax[1].set_title("Predicted")
ax[1].axis('off')

ax[2].imshow(resized_msk)
ax[2].set_title("Actual")
ax[2].axis('off')

# color value mapping
label_to_rgb = {
    'Undamaged': (51,255,255),
    'Minor Dmg': (255,255,0),
    'Major Dmg': (255,128,0),
    'Destroyed': (255,0,0),
}
# Creating patches for the legend with RGB colors
patches = [mpatches.Patch(color=np.array(rgb)/255, label=label) for label, rgb in label_to_rgb.items()]
# Adding the legend to the third subplot
ax[2].legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### Image from the internet 

In [ ]:
path = "Earth Quake Images/open_data_stat_block_img.jpg"
img = cv2.resize(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB), (512, 512), interpolation = cv2.INTER_LINEAR)
img = np.expand_dims(img, axis=0)

y_img = model.predict(img/255)

y_img = np.squeeze(y_img, axis=0)
y_img = np.argmax(y_img, axis=-1)
img = np.squeeze(img, axis=0)

y_img = y_img.astype(np.uint8)
y_img = cv2.cvtColor(y_img, cv2.COLOR_GRAY2RGB)
y_img[np.all(y_img == (1,1,1), axis=-1)] = (51,255,255)
y_img[np.all(y_img == (2,2,2), axis=-1)] = (255,255,0)
y_img[np.all(y_img == (3,3,3), axis=-1)] = (255,128,0)
y_img[np.all(y_img == (4,4,4), axis=-1)] = (255,0,0)

In [ ]:
fig3, ax = plt.subplots(1, 2)

ax[0].imshow(img)
ax[0].set_title("Image")
ax[0].axis('off')

ax[1].imshow(y_img)
ax[1].set_title("Predicted")
ax[1].axis('off')

patches = [mpatches.Patch(color=np.array(rgb)/255, label=label) for label, rgb in label_to_rgb.items()]

ax[1].legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
y_img_overlay = black_to_transparent(y_img) #

In [ ]:
y_img_overlay[np.all(y_img_overlay == (51,255,255,255), axis=-1)] = (51,255,255,20)
y_img_overlay[np.all(y_img_overlay == (255,255,0,255), axis=-1)] = (102,102,0,20)
y_img_overlay[np.all(y_img_overlay == (255,128,0,255), axis=-1)] = (102,51,0,20)
y_img_overlay[np.all(y_img_overlay == (255,0,0,255), axis=-1)] = (255,20,20 ,20)

In [ ]:
result = overlay_transparent(img.astype(np.uint8), y_img_overlay)

In [ ]:
# Creating patches for the legend with RGB colors
patches = [mpatches.Patch(color=np.array(rgb)/255, label=label) for label, rgb in label_to_rgb.items()]

# Adding the legend to the third subplot
plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.imshow(result)
plt.axis('off')

## Model Performance

In [ ]:
paths = ['../geotiffs/test/images', '../geotiffs/hold/images' ]

In [ ]:
image_files = []
for path in paths:
    image_files.extend([ os.path.join(path, file) for file in os.listdir(path) if file.endswith('.tif')])

# images_name = ['../geotiffs/test/images/hurricane-harvey_00000096_pre_disaster.tif']

### Using Cut Technique

In [ ]:
x_cut = np.empty((1,512,512,3), dtype=np.uint8)
y_cut = np.empty((1,512,512), dtype=np.uint8)

for i in tqdm(range(len(image_files[:100]))):
    
    images = cut_img(image_files[i])
    masks = cut_msk(image_files[i].replace('/images','/labels').replace('.tif','.json'))
        
    x_cut = np.concatenate([x_cut, images], axis=0)
    y_cut = np.concatenate([y_cut, masks], axis=0)

In [ ]:
model.evaluate(tf.cast(x_cut[1:], tf.float32)/255, tf.one_hot(y_cut[1:], depth=5))

### Using Resize Technique

In [ ]:
x_resize = np.zeros((1, 512,512,3), dtype=np.uint8)
y_resize = np.zeros((1, 512,512), dtype=np.uint8)

for i in tqdm(range(len(image_files[:200]))):
    
    image = np.expand_dims(cv2.resize(tifffile.imread(image_files[i]), (512, 512), interpolation = cv2.INTER_LINEAR), axis=0)
    mask = create_mask(image_files[i].replace('/images','/labels').replace('.tif','.json'))
    mask = np.expand_dims(cv2.resize(mask, (512, 512), interpolation = cv2.INTER_LINEAR), axis=0)
    
    x_resize = np.concatenate([x_resize, image], axis=0)
    y_resize = np.concatenate([y_resize, mask], axis=0)

In [ ]:
model.evaluate(tf.cast(x_resize[1:], tf.float32)/255, tf.one_hot(y_resize[1:], depth=5))